<a href="https://colab.research.google.com/github/Thabuki/iris-ml-mvp/blob/main/colab_ml_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classificação de Espécies de Íris

**Aluno:** Thales Rebelo  
**Disciplina:** Sprint: Qualidade de Software, Segurança e Sistemas Inteligentes  
**Curso:** Pós-graduação em Engenharia de Computação  

Este projeto foi desenvolvido como parte do MVP da disciplina.

O objetivo deste trabalho é aplicar técnicas de aprendizado de máquina para classificar flores do gênero íris em diferentes espécies, utilizando algoritmos clássicos de classificação como KNN, Árvore de Decisão, Naive Bayes e SVM.

Para este estudo, foi escolhido o dataset Iris, devido à sua simplicidade, organização e ampla utilização em contextos introdutórios de aprendizado de máquina. Trata-se de um conjunto de dados pequeno, bem estruturado e sem valores ausentes, o que facilita a compreensão das etapas do processo de modelagem. Além disso, por apresentar características numéricas e classes bem definidas, o dataset é adequado para a aplicação e comparação de diferentes algoritmos de classificação.

Trata-se de um projeto acadêmico, cujo foco está na aplicação prática dos conceitos estudados ao longo da disciplina, incluindo etapas de carga dos dados, pré-processamento, modelagem, avaliação de desempenho e comparação entre modelos.

Todas as etapas do processo serão documentadas ao longo deste notebook.

## Importação das Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

## Carregamento do Dataset

O dataset Iris foi carregado a partir de uma URL pública, permitindo a execução direta deste notebook no Google Colab sem necessidade de configuração manual adicional.

In [3]:
url = "https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv"
df = pd.read_csv(url)

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Separação de Variáveis (X e y)

Nesta etapa, separamos as variáveis independentes (X) da variável alvo (y). O modelo utilizará os dados de entrada (X) para prever a espécie da flor (y).

In [4]:
X = df.drop("species", axis=1)
y = df["species"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (150, 4)
y shape: (150,)


## Separação em Treino e Teste (Holdout)

Os dados foram divididos em conjuntos de treino e teste utilizando a técnica de holdout. O conjunto de treino será utilizado para treinar os modelos, enquanto o conjunto de teste será utilizado para avaliar seu desempenho.

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train_raw.shape)
print("X_test shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (120, 4)
X_test shape: (30, 4)
y_train shape: (120,)
y_test shape: (30,)


## Normalização dos Dados

Além da padronização, também foi realizada a normalização dos dados como etapa complementar de transformação. A normalização busca reescalar os valores para uma faixa comum, geralmente entre 0 e 1, o que pode ser útil em determinados algoritmos e contextos de comparação.

Neste trabalho, a normalização é apresentada para contemplar essa etapa do pré-processamento e evidenciar a diferença em relação à padronização, que foi a abordagem utilizada no treinamento principal dos modelos.

In [6]:
normalizer = MinMaxScaler()

X_train_norm = normalizer.fit_transform(X_train_raw)
X_test_norm = normalizer.transform(X_test_raw)

print("Exemplo de dados normalizados:")
print(X_train_norm[:3])

Exemplo de dados normalizados:
[[0.08823529 0.66666667 0.         0.04166667]
 [0.41176471 1.         0.0877193  0.125     ]
 [0.70588235 0.45833333 0.59649123 0.54166667]]


## Pré-processamento dos Dados

Nesta etapa, os dados foram padronizados utilizando o StandardScaler. Esse processo é importante para colocar as variáveis em uma escala comparável, o que pode melhorar o desempenho de algoritmos sensíveis à escala, como KNN e SVM.

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

## Otimização de Hiperparâmetros (KNN)

Nesta etapa, é realizada a otimização de hiperparâmetros do modelo KNN utilizando GridSearchCV. O objetivo é encontrar o valor ideal de "n_neighbors" que maximize o desempenho do modelo.

In [8]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9]
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Melhor parâmetro:", grid.best_params_)

Melhor parâmetro: {'n_neighbors': 3}


## Modelo 1: K-Nearest Neighbors (KNN)

O algoritmo KNN classifica uma amostra com base nos vizinhos mais próximos no espaço de características. É um método simples e eficaz para problemas de classificação.

In [9]:
knn = KNeighborsClassifier(n_neighbors=grid.best_params_["n_neighbors"])
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("KNN:", accuracy_score(y_test, y_pred_knn))

KNN: 1.0


## Validação Cruzada

Além da divisão entre treino e teste, foi utilizada validação cruzada para obter uma estimativa mais robusta do desempenho do modelo. Essa técnica reduz a dependência de uma única divisão dos dados e permite uma avaliação mais confiável.

In [10]:
cv_scores = cross_val_score(knn, X_train, y_train, cv=5)

print("Scores da validação cruzada:", cv_scores)
print("Média da validação cruzada:", cv_scores.mean())

Scores da validação cruzada: [0.95833333 1.         0.83333333 1.         0.95833333]
Média da validação cruzada: 0.95


## Modelo 2: Árvore de Decisão

A Árvore de Decisão é um modelo que toma decisões com base em regras aprendidas a partir dos dados, dividindo o espaço de características em regiões.

In [11]:
tree = DecisionTreeClassifier()
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("Decision Tree:", accuracy_score(y_test, y_pred_tree))

Decision Tree: 1.0


## Modelo 3: Naive Bayes

O Naive Bayes é um algoritmo probabilístico baseado no Teorema de Bayes. Ele assume independência entre as variáveis de entrada, o que simplifica o processo de classificação.

In [12]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes:", accuracy_score(y_test, y_pred_nb))

Naive Bayes: 1.0


## Modelo 4: Support Vector Machine (SVM)

O SVM é um algoritmo de classificação que busca encontrar a melhor fronteira de separação entre as classes. Ele é especialmente útil em problemas com boa separação entre categorias.

In [13]:
svm = SVC()
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("SVM:", accuracy_score(y_test, y_pred_svm))

SVM: 1.0


## Comparação dos Resultados

Nesta etapa, os resultados obtidos pelos diferentes algoritmos são comparados com base na métrica de acurácia, permitindo identificar o desempenho de cada modelo no problema proposto.

In [14]:
results = {
    "KNN": accuracy_score(y_test, y_pred_knn),
    "Decision Tree": accuracy_score(y_test, y_pred_tree),
    "Naive Bayes": accuracy_score(y_test, y_pred_nb),
    "SVM": accuracy_score(y_test, y_pred_svm),
}

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

KNN: 1.0000
Decision Tree: 1.0000
Naive Bayes: 1.0000
SVM: 1.0000


## Análise de Resultados

Os resultados obtidos mostram que todos os modelos avaliados alcançaram alta acurácia no conjunto de teste utilizado, evidenciando a boa separação entre as classes no dataset Iris.

Além da avaliação com holdout, foi realizada validação cruzada no modelo KNN, obtendo uma média de desempenho próxima de 0.95. Esse resultado reforça a consistência do modelo e indica que seu desempenho não depende apenas de uma única divisão dos dados.

De modo geral, observou-se que diferentes algoritmos apresentaram desempenho semelhante neste problema, o que era esperado devido à simplicidade e organização do dataset. Ainda assim, os resultados demonstram a importância do pré-processamento e da escolha adequada de hiperparâmetros no desempenho final dos modelos.

## Exportação do Modelo

Após a comparação dos modelos, o modelo selecionado será exportado para utilização posterior na aplicação full stack. Além do modelo, também será salvo o objeto de padronização, para garantir que novos dados recebam o mesmo tratamento aplicado durante o treinamento.

Embora todos os modelos tenham apresentado desempenho máximo no conjunto de teste, o modelo SVM foi selecionado para exportação e integração com a aplicação full stack.

In [15]:
joblib.dump(svm, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

## Reflexão sobre Boas Práticas de Software Seguro

Embora o dataset Iris não contenha dados pessoais sensíveis, é importante refletir sobre como boas práticas de segurança poderiam ser aplicadas em problemas reais de machine learning.

Caso o projeto utilizasse dados de pessoas ou informações confidenciais, seria necessário adotar técnicas de anonimização para remover ou proteger identificadores diretos e indiretos. Além disso, seria importante aplicar controles de acesso, validação de entradas, proteção no armazenamento dos arquivos do modelo e cuidado com a exposição de informações sensíveis por meio da aplicação.

No contexto da aplicação full stack deste projeto, também é relevante garantir que os dados enviados pelo usuário sejam validados adequadamente no back-end, reduzindo riscos de uso indevido, entradas inválidas e comportamentos inesperados.

Dessa forma, mesmo em um projeto acadêmico e com dados não sensíveis, as práticas de desenvolvimento seguro continuam sendo relevantes como referência para aplicações reais.

## Conclusão

Neste trabalho, foi desenvolvido um processo completo de classificação utilizando o dataset Iris e algoritmos clássicos de aprendizado de máquina.

Foram realizadas as etapas de carregamento dos dados, separação entre treino e teste, pré-processamento com padronização, treinamento de diferentes modelos e comparação dos resultados com base na acurácia.

Os resultados mostraram que todos os modelos avaliados obtiveram desempenho máximo no conjunto de teste utilizado, evidenciando que o problema proposto possui boa separação entre classes e é adequado para fins de estudo e aplicação introdutória de técnicas de machine learning.

Por fim, o modelo escolhido foi exportado para posterior integração com a aplicação full stack prevista no escopo do MVP.